# 🎓 Modelo de Machine Learning — Previsão de Evasão (Pós-Graduação EAD)
### a instituicao | v3 — validação em origem deslizante

Prevê a probabilidade de um ingressante da pós-graduação EAD **sair sem concluir**
dentro de 180 dias, para que a equipe de retenção priorize quem abordar.

O modelo não existe para acertar uma métrica: ele existe para **ordenar uma fila**.
Como a base é pequena, a métrica que manda aqui é **`recall@10%`** — dos evasores
reais, quantos caem no topo 10% da lista — e não `recall@200`, que cobriria 26%
desta base e não seria comparável com as outras modalidades.

**Configuração, e por que ela é essa** (medido em 18/09/2026, origem deslizante mensal):

| Decisão | Motivo |
|---|---|
| Alvo por **horizonte em dias**, não por período letivo | O calendário da pós virou **anual** em 2025 (período `6`); o "período seguinte" cairia em 2027 e não existe |
| Observação de **30 dias** | 60 dias derruba os evasores de 34 para 17; 90 dias, para 13 — o cancelamento é precoce |
| Só quem **efetivamente iniciou** (`@EXIGE_CONTA_AVA=1`) | 74 dos 75 `Inconcluido` não têm conta no AVA: mantê-los entrega 69% do alvo de graça |
| Sem imputação de mediana em `AVA_*` | nulo é informação — significa "não acessou"; o RF do sklearn ≥ 1.4 trata NaN |
| Sem bloco acadêmico | só 19 de 976 alunos têm disciplina encerrada dentro da janela |
| Limiar escolhido **na validação** | a versão anterior escolhia olhando o próprio teste |

> ### ⚠️ Leia antes de usar o resultado
>
> Este notebook **substitui integralmente** a versão anterior, que reportava ROC-AUC de
> **0,948**. Aquele número era vazamento: sem `DT_CORTE`, as features agregavam a vida
> inteira do aluno, o alvo saía da situação cadastral atual e 62% da base eram alunos
> ainda em curso contados como não-evasão.
>
> Com a medição correta, **não há modelo publicável nesta rodada**. O ranqueador honesto
> alcança `recall@10%` de 27,3% contra 10,1% do acaso (p-valor empírico 0,017), o que o
> coloca acima do ruído — mas são **6 acertos sobre 22 evasores**, e, medida no mesmo
> conjunto, uma regra de **um único critério** ("ordene por dias ativos no AVA") entrega
> exatamente os mesmos 6. O modelo não agrega nada sobre ela. O gargalo é dado, não método.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["font.sans-serif"] = "Arial"
plt.rcParams["font.family"] = "sans-serif"
sns.set_palette("muted")
COR_INST = "#1A365D"
pd.set_option("display.width", 120)

## 1. Carregamento dos Dados

Base gerada por `querys/06_extracao_pos_ead.sql`, com os parâmetros no topo do arquivo
(`@DIAS_OBS = 30`, `@DIAS_HORIZONTE = 180`, `@DT_INI_POP = '2025-08-01'`,
`@EXIGE_CONTA_AVA = 1`).

Lemos o **CSV**, não o XLSX: o Excel já converteu código de `CURSO` em data e corrompeu
acentos em extrações passadas.

In [ ]:
CSV = "../../tabelas/extracao_pos_ead.csv"
if not os.path.exists(CSV):
    CSV = "tabelas/extracao_pos_ead.csv"

print(f"Carregando: {CSV}")
df = pd.read_csv(
    CSV,
    encoding="utf-8",
    dtype={"ALUNO": str, "CURSO": str, "END_MUNICIPIO": str},
)
df["DT_INGRESSO"] = pd.to_datetime(df["DT_INGRESSO"], format="ISO8601")
print(f"Dimensões brutas: {df.shape[0]:,} linhas x {df.shape[1]} colunas\n")
print(f"Janela do AVA respeitada: ingressos de "
      f"{df.DT_INGRESSO.min():%d/%m/%Y} a {df.DT_INGRESSO.max():%d/%m/%Y}")

## 2. Filtro Obrigatório de Elegibilidade (`ELEGIVEL == 1`)

Descarta duas situações. A primeira é quem ainda **não chegou ao fim do horizonte** de
180 dias — sem isso, aluno que ainda pode evadir entra como permanente, que é a censura
à direita que envenenou a base do v1. A segunda é quem já havia concluído, cancelado ou
sido transferido **antes** do corte: não há o que prever.

Criamos também `MES_INGRESSO`, que faz o papel de `COORTE` das outras modalidades — na
pós o ingresso é contínuo, então a origem deslizante é mensal.

In [ ]:
df_model = df[df["ELEGIVEL"] == 1].copy()
df_model = df_model.sort_values("DT_INGRESSO").reset_index(drop=True)
df_model["MES_INGRESSO"] = df_model["DT_INGRESSO"].dt.to_period("M")

y_all = df_model["EVADIU"].to_numpy()

print(f"Elegíveis: {len(df_model):,}  |  Evasores: {int(y_all.sum()):,}"
      f"  |  Taxa: {y_all.mean():.2%}\n")

resumo = (
    df_model.groupby("MES_INGRESSO")
    .agg(ALUNOS=("ALUNO", "count"), EVASORES=("EVADIU", "sum"))
    .assign(TAXA=lambda d: (100 * d.EVASORES / d.ALUNOS).round(2))
)
print(resumo.to_string())
print(f"\nDescartados pelo filtro: {len(df) - len(df_model):,} alunos "
      f"({1 - len(df_model) / len(df):.1%}) — horizonte aberto ou saída anterior ao corte.")

## 3. Engenharia de Features de Risco Combinado

Deliberadamente **enxuta**, e essa é a decisão de projeto mais importante deste tópico.

Com 34 evasores, cada feature nova custa graus de liberdade que a base não tem: a razão
de positivos por variável já está em 1,7, contra as ~10 que a prática recomenda. As
razões compostas que a versão anterior criava (`SESSOES_POR_DIA_ATIVO`,
`RATIO_GAP_ATIVOS`, `PROP_TITULOS_EM_ABERTO`) eram quase colineares com seus
componentes e não pagavam o custo.

O que a query já entrega pronto e **é** engenharia de feature:

- `AVA_SESSOES_1A_METADE` e `AVA_SESSOES_2A_METADE` — trajetória dentro da janela.
- `AVA_DIAS_DESDE_ULT_ACESSO` — distância do último acesso até o corte.
- `AVA_MAIOR_GAP_DIAS` — maior silêncio dentro da observação.
- `AVA_PCT_SESSOES_APP` — canal (AVA-app/app contra AVA-web/web).

In [ ]:
# Nenhuma feature derivada e criada aqui, por decisao medida (ver texto acima).
# Este bloco existe para registrar a verificacao que sustenta a decisao.
pares = [
    ("AVA_QTD_SESSOES", "AVA_DIAS_ATIVOS"),
    ("AVA_SESSOES_1A_METADE", "AVA_QTD_SESSOES"),
    ("FIN_QTD_EM_ABERTO", "FIN_QTD_TITULOS"),
]
print("Colinearidade entre candidatas a razao composta e seus componentes:")
for a, b in pares:
    r = df_model[a].corr(df_model[b])
    print(f"  {a:26s} x {b:22s}  r = {r:+.3f}")

print("\nPor que nao criar razoes: com 34 positivos, uma feature que carrega")
print("a mesma informacao de duas colunas ja presentes so adiciona variancia.")

## 4. Separação de $X$ (Features) e $y$ (Target)

Colunas banidas do `X`, cada uma por um motivo medido:

| Coluna | Motivo |
|---|---|
| `EVADIU`, `ELEGIVEL` | é o alvo |
| `SIT_ALVO`, `DT_SAIDA` | derivam do alvo; existem para auditoria |
| `ALUNO` | identificador — o modelo decora o aluno |
| `DT_CORTE`, `DT_ALVO`, `PARAM_*` | auditoria da janela |
| **`AVA_TEM_CONTA`** | **vazamento**: 74 dos 75 `Inconcluido` caem aqui |
| `SEXO` | ressalva ética — só auditoria de viés (tópico 15) |
| `NOME_CURSO`, `FACULDADE`, `MODALIDADE_CURSO` | texto redundante com `CURSO` |

`AVA_TEM_CONTA` é o `NAO_TEVE_2O_SEMESTRE` do v1 reencarnado: sozinha entregaria 69%
do alvo sem que o modelo aprendesse nada.

In [ ]:
FEATURES = [
    # --- AVA: janela [DT_INGRESSO, DT_CORTE) ---
    "AVA_UTILIZOU", "AVA_QTD_SESSOES", "AVA_DIAS_ATIVOS", "AVA_DIAS_ATE_1O_ACESSO",
    "AVA_DIAS_DESDE_ULT_ACESSO", "AVA_PCT_FIM_DE_SEMANA", "AVA_PCT_NOTURNO",
    "AVA_PCT_SESSOES_APP", "AVA_SESSOES_1A_METADE", "AVA_SESSOES_2A_METADE",
    "AVA_MAIOR_GAP_DIAS",
    # --- Financeiro: so vencimentos ate DT_CORTE ---
    "FIN_QTD_TITULOS", "FIN_VALOR_MEDIO_PARCELA", "FIN_QTD_EM_ABERTO",
    "FIN_QTD_PAGO_ATRASO", "FIN_QTD_PAGO_EM_DIA", "FIN_MAX_DIAS_ATRASO_PAGO",
    "FIN_MEDIA_DIAS_ATRASO",
    # --- Cadastro ---
    "BOLSA_PERC", "IDADE_INGRESSO",
]
COLUNAS_REMOVER = [
    "ALUNO", "CURSO", "NOME_CURSO", "MODALIDADE_CURSO", "FACULDADE", "ANO_INGRESSO",
    "PER_INGRESSO", "DT_INGRESSO", "DT_CORTE", "DT_ALVO", "PARAM_DIAS_OBS",
    "PARAM_DIAS_HORIZONTE", "SIT_ALVO", "DT_SAIDA", "UNIDADE_FISICA", "TURNO",
    "TIPO_INGRESSO", "EVADIU", "ELEGIVEL", "EST_CIVIL", "END_MUNICIPIO", "SEXO",
    "AVA_TEM_CONTA",
]

X_all = df_model[FEATURES].to_numpy(dtype=float)
y_all = df_model["EVADIU"].to_numpy()

print(f"Matriz X: {X_all.shape[0]:,} linhas x {X_all.shape[1]} colunas")
print(f"Vetor y:  {y_all.shape[0]:,} registros  |  positivos: {int(y_all.sum())}")
print(f"\nPositivos por feature: {y_all.sum() / X_all.shape[1]:.1f}  "
      f"(abaixo de 10 e sinal de alerta — ver topico 9)")

## 5. Encoding de Categóricas — **sem imputação**

Não há categórica no `X`: `CURSO` e `END_MUNICIPIO` foram testados agrupados e não
pagaram (o `recall@10%` caiu de 27,3% para 22,7%), então ficaram de fora.

**O nulo não é imputado, e isso é decisão medida.** Em `AVA_*` o nulo significa "não
acessou o AVA na janela" — quase o próprio sinal de evasão. Em `FIN_*` significa "sem
título vencido até o corte". Substituir por mediana apaga exatamente o que o modelo
precisa ver. A versão anterior usava `SimpleImputer(strategy='median')` em tudo.

O `RandomForest` do sklearn ≥ 1.4 trata `NaN` nativamente. Só a regressão logística
precisa de imputação — e ali ela vem com `add_indicator=True`, que preserva a
informação de ausência numa coluna própria.

In [ ]:
nulos = df_model[FEATURES].isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)

print("Nulos preservados (NAO imputados globalmente):")
for col, n in nulos.items():
    print(f"  {col:28s} {n:4d}  ({n / len(df_model):5.1%})")

print(f"\nsklearn: RandomForest trata NaN nativamente.")
print(f"Para a regressao logistica, o pipeline usa "
      f"SimpleImputer(add_indicator=True) -> a ausencia vira feature.")

## 6. Divisão Temporal — treino no passado, teste na turma seguinte

**Split temporal, nunca aleatório.** Split aleatório infla a métrica num problema
temporal, e é assim que o modelo será usado: treinado no passado, aplicado na turma nova.

Como o ingresso na pós é contínuo, a origem deslizante é **mensal**: cada origem treina
em `ingresso < mês` e testa em `ingresso == mês`. As três primeiras ficam de fora por
não terem positivos suficientes para treinar.

A última origem é reservada: o limiar do tópico 8 é escolhido **sem olhar para ela**.

In [ ]:
MESES = sorted(df_model["MES_INGRESSO"].unique())
ORIGENS = MESES[3:]
ULTIMA_ORIGEM = str(MESES[-1])

print(f"Meses disponiveis: {len(MESES)}  ({MESES[0]} a {MESES[-1]})")
print(f"Origens avaliadas: {len(ORIGENS)}  (as 3 primeiras ficam so como treino)")
print(f"Ultima origem (reservada): {ULTIMA_ORIGEM}\n")

for m in ORIGENS:
    tr = (df_model["MES_INGRESSO"] < m).to_numpy()
    te = (df_model["MES_INGRESSO"] == m).to_numpy()
    print(f"  origem {m}: treino {tr.sum():3d} ({int(y_all[tr].sum()):2d} evasores)"
          f"  ->  teste {te.sum():3d} ({int(y_all[te].sum()):2d} evasores)")

## 7. Treinamento do Modelo (Sensível a Custo)

Base desbalanceada (4,4% de evasão), então `class_weight="balanced"` nos dois candidatos.

Testamos **dois modelos**, e a escolha não é a usual. Com 34 positivos e 20 features, o
Random Forest decora o treino: fica em `recall@10%` de 13,6%, **abaixo** da regra de um
critério só. A regressão logística com regularização L2 forte (`C=0.1`) é o que resta de
pé — menos capacidade, menos variância, e a única acima do ruído.

In [ ]:
faz_rf = lambda: RandomForestClassifier(
    n_estimators=400, max_depth=4, min_samples_leaf=15,
    class_weight="balanced", random_state=42, n_jobs=-1,
)
faz_lr = lambda: make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    StandardScaler(),
    LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000),
)


def recall_em_k(y_true, proba, frac=0.10):
    """Dos evasores reais, quantos caem no topo `frac` da fila."""
    k = max(1, int(round(len(y_true) * frac)))
    ordem = np.argsort(-proba)[:k]
    return y_true[ordem].sum() / max(1, y_true.sum()), k, int(y_true[ordem].sum())


def origem_deslizante(X, y, meses, make_model, nome, verbose=True):
    oof = np.full(len(y), np.nan)
    origem_de = np.full(len(y), "", dtype=object)
    linhas = []
    for m in ORIGENS:
        tr = (meses < m).to_numpy()
        te = (meses == m).to_numpy()
        if y[tr].sum() < 3 or te.sum() == 0:
            continue
        modelo = make_model()
        modelo.fit(X[tr], y[tr])
        p = modelo.predict_proba(X[te])[:, 1]
        oof[te] = p
        origem_de[te] = str(m)
        if y[te].sum() > 0:
            r, _, h = recall_em_k(y[te], p)
            linhas.append({
                "Origem": str(m), "Teste": int(te.sum()), "Evasores": int(y[te].sum()),
                "PR-AUC": round(average_precision_score(y[te], p), 4),
                "recall@10%": f"{r:.1%}",
            })
    ok = ~np.isnan(oof)
    r, k, h = recall_em_k(y[ok], oof[ok])
    if verbose:
        print(f"--- {nome} ---")
        print(pd.DataFrame(linhas).to_string(index=False))
        print(f"\nOOF: PR-AUC {average_precision_score(y[ok], oof[ok]):.4f} | "
              f"ROC-AUC {roc_auc_score(y[ok], oof[ok]):.3f} | "
              f"recall@10% {r:.1%} ({h} de {int(y[ok].sum())} no topo {k})\n")
    return oof, origem_de


meses_s = df_model["MES_INGRESSO"]
oof_rf, _ = origem_deslizante(X_all, y_all, meses_s, faz_rf, "Random Forest (NaN nativo)")
oof_lr, origem_de = origem_deslizante(X_all, y_all, meses_s, faz_lr, "Regressao Logistica L2 (C=0.1)")

y_proba = oof_lr                      # modelo adotado
OK = ~np.isnan(y_proba)
y_test = y_all[OK]
y_score = y_proba[OK]
print(f"Conjunto de avaliacao cega: {OK.sum()} alunos | {int(y_test.sum())} evasores")

## 8. O Limiar de Decisão e o Tamanho da Fila

O limiar é escolhido **nas origens de validação** (todas menos a última) e só depois
aplicado ao conjunto completo. A versão anterior lia a tabela de limiares no próprio
conjunto de teste e reportava o mesmo conjunto — o que garante número bonito e sem valor.

A coluna decisiva é **`Precisão`**: ela fica entre 5% e 11% em toda a faixa. O limiar
troca volume de fila por cobertura, sem que a qualidade do alerta melhore. Como no
híbrido, **a ordem da lista importa mais que o corte**.

In [ ]:
val = OK & (origem_de != ULTIMA_ORIGEM)

linhas = []
for t in [0.80, 0.70, 0.60, 0.50, 0.40, 0.30]:
    yp = (y_proba[val] >= t).astype(int)
    rec = recall_score(y_all[val], yp, zero_division=0)
    pre = precision_score(y_all[val], yp, zero_division=0)
    f1 = 0.0 if (rec + pre) == 0 else 2 * rec * pre / (rec + pre)
    linhas.append({
        "Limiar": f"{t:.2f}", "Recall": f"{rec:.1%}", "Precisão": f"{pre:.1%}",
        "F1": f"{f1:.3f}", "Alertas": int(yp.sum()),
        "Evasores pegos": f"{int(yp[y_all[val] == 1].sum())}/{int(y_all[val].sum())}",
    })

tab_lim = pd.DataFrame(linhas)
print(f"Escolha feita nas origens de VALIDACAO "
      f"({val.sum()} alunos, {int(y_all[val].sum())} evasores) — "
      f"a origem {ULTIMA_ORIGEM} fica de fora.\n")
print(tab_lim.to_string(index=False))

LIMIAR = 0.60
print(f"\nLimiar adotado: {LIMIAR:.2f} (maior F1 na validacao)")

## 9. Avaliação no Limiar 0,60

Três números, nesta ordem de importância:

1. **`recall@10%`** — dos evasores reais, quantos caem no topo 10% da fila.
2. **PR-AUC** — qualidade geral do ranking.
3. **`precision@K`** — quanto esforço da equipe é desperdiçado.

A acurácia não entra em nenhuma decisão: com 4,4% de evasão, chutar "ninguém evade" dá
95,6% de acurácia e serve para nada.

**A régua que falta nos outros notebooks.** Com uma base tão desbalanceada, o número só
significa alguma coisa contra duas referências, e as duas precisam ser medidas **no mesmo
conjunto** que o modelo:

1. **O acaso.** Ordenar aleatoriamente já acerta ~10% dos evasores no topo 10%. Simulamos
   2.000 ordenações para ter a distribuição e o p-valor empírico.
2. **A regra de um critério só.** Ordenar por `AVA_DIAS_ATIVOS`, sem modelo nenhum. Se o
   modelo não superar isso, ele não se justifica — carrega 20 variáveis e um pipeline para
   entregar o que uma linha de SQL entrega.

In [ ]:
y_pred = (y_score >= LIMIAR).astype(int)

print("=" * 70)
print(f"   RELATÓRIO DE CLASSIFICAÇÃO — limiar {LIMIAR:.2f} — avaliação cega")
print("=" * 70)
print(classification_report(y_test, y_pred,
                            target_names=["Permanência (0)", "Evasão (1)"],
                            zero_division=0))

roc_auc = roc_auc_score(y_test, y_score)
pr_auc = average_precision_score(y_test, y_score)
r10, k10, h10 = recall_em_k(y_test, y_score)

rng = np.random.default_rng(0)
sorteios = [recall_em_k(y_test, rng.random(len(y_test)))[0] for _ in range(2000)]
p95 = float(np.percentile(sorteios, 95))

baseline_ava = -np.nan_to_num(df_model["AVA_DIAS_ATIVOS"].to_numpy(dtype=float)[OK], nan=-1)
r_regra = recall_em_k(y_test, baseline_ava)[0]

print(f"ROC-AUC ............ {roc_auc:.4f}")
print(f"PR-AUC ............. {pr_auc:.4f}")
print(f"recall@10% ......... {r10:.1%}  ({h10} de {int(y_test.sum())} no topo {k10})")
print(f"precision@{k10} ......... {y_test[np.argsort(-y_score)[:k10]].mean():.1%}")
print()
print("=" * 70)
print("   A RÉGUA: O QUE O ACASO ENTREGA")
print("=" * 70)
sorteios = np.asarray(sorteios)
pval = lambda v: float((sorteios >= v).mean())
r_rf = recall_em_k(y_all[~np.isnan(oof_rf)], oof_rf[~np.isnan(oof_rf)])[0]

comp = pd.DataFrame([
    {"Ranqueador": "Acaso (média de 2.000 sorteios)", "recall@10%": f"{sorteios.mean():.1%}",
     "p-valor": "—", "Leitura": "piso absoluto"},
    {"Ranqueador": "Acaso (percentil 95)", "recall@10%": f"{p95:.1%}",
     "p-valor": "0,050", "Leitura": "limite do ruído"},
    {"Ranqueador": "Random Forest", "recall@10%": f"{r_rf:.1%}",
     "p-valor": f"{pval(r_rf):.3f}", "Leitura": "dentro do ruído"},
    {"Ranqueador": 'Regra: "menos dias ativos no AVA"', "recall@10%": f"{r_regra:.1%}",
     "p-valor": f"{pval(r_regra):.3f}", "Leitura": "um critério só, SEM modelo"},
    {"Ranqueador": "Regressão Logística L2 (adotado)", "recall@10%": f"{r10:.1%}",
     "p-valor": f"{pval(r10):.3f}", "Leitura": f"{h10} acertos — empata com a regra"},
])
print(comp.to_string(index=False))
print()
print("=" * 70)
print("   O RESULTADO QUE DECIDE O PROJETO")
print("=" * 70)
print(f"Modelo e regra simples acertam exatamente os MESMOS {h10} evasores de "
      f"{int(y_test.sum())}.")
print("Ambos ficam acima do acaso (p = 0,017), entao ha sinal real na atividade")
print("do AVA. Mas o modelo NAO agrega nada sobre ordenar por uma unica coluna:")
print("20 variaveis e um pipeline para entregar o que uma linha de SQL entrega.")
print("E isso, e nao a fragilidade estatistica, que desaconselha publicar o modelo.")

## 10. Matrizes de Confusão Comparativas (0,50 vs 0,60)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, t, cmap, rot in [
    (axes[0], 0.50, "Blues", "Limiar padrão (0,50)"),
    (axes[1], LIMIAR, "Greens", f"Limiar do projeto ({LIMIAR:.2f})"),
]:
    yp = (y_score >= t).astype(int)
    cm = confusion_matrix(y_test, yp, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, ax=ax, cbar=False,
                xticklabels=["Previsto: Fica", "Previsto: Evade"],
                yticklabels=["Real: Fica", "Real: Evade"])
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f"{rot} — pega {tp} de {tp + fn} evasores, "
                 f"fila de {tp + fp}",
                 fontsize=11, fontweight="bold", color=COR_INST)
    ax.set_xlabel("Predição do modelo", fontweight="bold")
    ax.set_ylabel("Realidade", fontweight="bold")

plt.tight_layout()
plt.show()

## 11. Curva ROC e Curva Precision-Recall

Numa base com 4,4% de positivos, a **curva Precision-Recall é a que informa**. A ROC
parece sempre melhor do que a realidade quando a classe positiva é rara, porque a taxa
de falsos positivos tem um denominador enorme.

A linha tracejada vermelha é a precisão de um classificador aleatório — a distância
entre a curva e essa linha é o valor real que o modelo agrega.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fpr, tpr, _ = roc_curve(y_test, y_score)
axes[0].plot(fpr, tpr, color=COR_INST, lw=2.5, label=f"Reg. Logística (AUC = {roc_auc:.3f})")
axes[0].plot([0, 1], [0, 1], color="gray", linestyle="--", label="Acaso")
axes[0].set_title("Curva ROC — Pós-Graduação EAD", fontsize=12, fontweight="bold", color=COR_INST)
axes[0].set_xlabel("Taxa de Falsos Positivos (1 - Especificidade)")
axes[0].set_ylabel("Taxa de Verdadeiros Positivos (Recall)")
axes[0].legend(loc="lower right")

prec, rec, _ = precision_recall_curve(y_test, y_score)
axes[1].plot(rec, prec, color="#2B6CB0", lw=2.5, label=f"Reg. Logística (PR-AUC = {pr_auc:.3f})")
axes[1].axhline(y=y_test.mean(), color="#C53030", linestyle="--",
                label=f"Acaso ({y_test.mean():.1%})")
axes[1].set_title("Curva Precision-Recall — Pós-Graduação EAD", fontsize=12,
                  fontweight="bold", color=COR_INST)
axes[1].set_xlabel("Recall (Sensibilidade)")
axes[1].set_ylabel("Precisão")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

## 12. Importância das Features

Na regressão logística a importância é o **coeficiente padronizado**: como todas as
features passaram pelo `StandardScaler`, os coeficientes são diretamente comparáveis.
Coeficiente positivo empurra para evasão; negativo, para permanência.

Preferimos isso ao Gini do Random Forest, que é enviesado para variáveis contínuas de
alta cardinalidade — no caso, justamente as contagens do AVA.

In [ ]:
modelo_final = faz_lr()
modelo_final.fit(X_all, y_all)

imputer_f = modelo_final.named_steps["simpleimputer"]
logreg_f = modelo_final.named_steps["logisticregression"]

nomes = list(FEATURES) + [f"{FEATURES[i]}__AUSENTE" for i in imputer_f.indicator_.features_]
coefs = logreg_f.coef_[0]

feat_df = (pd.DataFrame({"Feature": nomes, "Coeficiente": coefs})
           .assign(Magnitude=lambda d: d.Coeficiente.abs())
           .sort_values("Magnitude", ascending=False))

top = feat_df.head(15).iloc[::-1]
cores = ["#C53030" if v > 0 else "#2B6CB0" for v in top.Coeficiente]

plt.figure(figsize=(11, 7))
plt.barh(top.Feature, top.Coeficiente, color=cores, edgecolor="black", linewidth=0.5)
plt.axvline(0, color="gray", linestyle="--", linewidth=0.8)
plt.title("Top 15 Features — Pós EAD (coeficiente padronizado)\n"
          "vermelho empurra para evasão · azul para permanência",
          fontsize=12, fontweight="bold", color=COR_INST)
plt.xlabel("Coeficiente (escala padronizada)", fontweight="bold")
plt.tight_layout()
plt.show()

print(feat_df.head(10).round(4).to_string(index=False))

## 13. Salvamento do Artefato (`.pkl`)

O artefato guarda o **pipeline completo** (imputação com indicador → padronização →
regressão logística), as features, o limiar e as métricas out-of-fold.

Guarda também um campo `aviso`, que é lido e impresso pelos notebooks de
`area-testes/pos-ead/`: quem carregar este modelo precisa ver, antes de qualquer número,
que o sinal é marginal.

Salvamos junto o `_oof_scores.csv` — as predições out-of-fold por aluno. É ele que a
bateria de testes consome, para que a avaliação não seja feita sobre a base de treino.

In [ ]:
metricas = {}
for nome, msk in [("validacao", val), ("ultima_origem", OK & (origem_de == ULTIMA_ORIGEM)),
                  ("oof_total", OK)]:
    if y_all[msk].sum() == 0:
        continue
    r, k, h = recall_em_k(y_all[msk], y_proba[msk])
    yp = (y_proba[msk] >= LIMIAR).astype(int)
    metricas[nome] = {
        "n": int(msk.sum()), "evasores": int(y_all[msk].sum()),
        "pr_auc": float(average_precision_score(y_all[msk], y_proba[msk])),
        "roc_auc": float(roc_auc_score(y_all[msk], y_proba[msk])),
        "recall_at_10pct": float(r), "topo_k": int(k), "acertos_topo": int(h),
        "recall_thresh": float(recall_score(y_all[msk], yp, zero_division=0)),
        "precisao_thresh": float(precision_score(y_all[msk], yp, zero_division=0)),
        "fila_thresh": int(yp.sum()),
    }

artifact = {
    "model": modelo_final,
    "imputer": None,                      # embutido no pipeline
    "feature_names": FEATURES,
    "colunas_remover": COLUNAS_REMOVER,
    "threshold": LIMIAR,
    "optimal_threshold": LIMIAR,
    "principais_municipios": [],
    "metadata": {
        "modalidade": "POS_EAD",
        "cenario": "B (@EXIGE_CONTA_AVA=1) - so alunos com conta no AVA",
        "query": "querys/06_extracao_pos_ead.sql",
        "dias_obs": 30, "dias_horizonte": 180,
        "coorte": f"ingressos {df_model.DT_INGRESSO.min():%Y-%m-%d} a "
                  f"{df_model.DT_INGRESSO.max():%Y-%m-%d}",
        "n_elegiveis": int(len(df_model)), "n_evasores": int(y_all.sum()),
        "taxa_evasao": float(y_all.mean()),
        "validacao": "origem deslizante mensal por DT_INGRESSO",
        "metricas_oof": metricas,
        "aviso": ("Modelo de sinal MARGINAL. recall@10% out-of-fold de 27,3% (6 de 22) "
                  "contra 10,1% do acaso, p-valor 0,017. Porem a regra de um criterio so "
                  "-- ordenar por AVA_DIAS_ATIVOS -- acerta os MESMOS 6 evasores no mesmo "
                  "conjunto. O modelo nao agrega sobre ela. Prefira a regra: e auditavel "
                  "e explicavel. NAO usar este .pkl como unico criterio de priorizacao."),
    },
}

destinos = ["modelo_evasao_pos_ead.pkl",
            os.path.join("..", "..", "area-testes", "pos-ead", "modelo_evasao_pos_ead.pkl")]
for d in destinos:
    if os.path.isdir(os.path.dirname(d)) or os.path.dirname(d) == "":
        joblib.dump(artifact, d)
        print(f"Artefato salvo: {d}")

oof_out = pd.DataFrame({"ALUNO": df_model["ALUNO"], "OOF": y_proba, "ORIGEM": origem_de})
dir_testes = os.path.join("..", "..", "area-testes", "pos-ead")
if os.path.isdir(dir_testes):
    oof_out.to_csv(os.path.join(dir_testes, "_oof_scores.csv"), index=False)
    print(f"Scores out-of-fold salvos: {dir_testes}/_oof_scores.csv")

## 14. Matriz de Correlação de Pearson

Mesmo padrão visual das outras modalidades (`relatorios/matriz-correlacao/`): triângulo
inferior, `vlag` centralizado em zero, ordenado pela correlação com `EVADIU`.

Serve para ler a estrutura das features, **não** para selecionar feature — Pearson só
captura relação linear, e várias colunas aqui são contagens truncadas em zero.

Uma diferença em relação à versão anterior: a matriz é calculada **preservando os
nulos** (`pairwise`), não sobre `dropna()`. Com `dropna()` ela sairia de uma subamostra
de alunos que têm AVA **e** financeiro **e** bolsa, que não é a base em que o modelo treina.

In [ ]:
from pathlib import Path

dir_saida = Path("../../relatorios/matriz-correlacao")
dir_saida.mkdir(parents=True, exist_ok=True)

corr = df_model[FEATURES + ["EVADIU"]].corr(method="pearson", min_periods=30)
ordem = corr["EVADIU"].sort_values().index
corr = corr.loc[ordem, ordem]

plt.figure(figsize=(13, 11), dpi=150)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="vlag", vmax=0.6, vmin=-0.6, center=0,
            square=True, linewidths=0.5, annot=True, fmt=".2f",
            cbar_kws={"shrink": 0.8, "label": "Correlação de Pearson (r)"},
            annot_kws={"size": 7.5, "weight": "bold"})
plt.title(f"Matriz de Correlação de Pearson — Pós-Graduação EAD\n"
          f"(N = {len(df_model):,} elegíveis | {int(y_all.sum())} evasores | "
          f"janela 30d, horizonte 180d)",
          fontsize=13, fontweight="bold", pad=15, color=COR_INST)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(dir_saida / "matriz_correlacao_pos_ead.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
ranking = corr["EVADIU"].drop("EVADIU").sort_values()
sel = pd.concat([ranking.head(8), ranking.tail(8)])
cores = ["#2B6CB0" if v < 0 else "#C53030" for v in sel.values]

plt.figure(figsize=(12, 6), dpi=150)
barras = plt.barh(sel.index, sel.values, color=cores, edgecolor="black", linewidth=0.5)
plt.axvline(0, color="gray", linestyle="--", linewidth=0.8)
plt.title("Correlação linear com a evasão — Pós EAD (EVADIU)",
          fontsize=12, fontweight="bold", pad=15, color=COR_INST)
plt.xlabel("Coeficiente de Pearson (r)", fontsize=10, fontweight="bold")
plt.ylabel("Feature", fontsize=10, fontweight="bold")
for b in barras:
    w = b.get_width()
    plt.text(w + (0.004 if w >= 0 else -0.004), b.get_y() + b.get_height() / 2,
             f"{w:+.2f}", va="center", ha="left" if w >= 0 else "right",
             fontsize=9, fontweight="bold", color="#2D3748")
plt.xlim(sel.values.min() - 0.05, sel.values.max() + 0.05)
plt.tight_layout()
plt.savefig(dir_saida / "ranking_correlacao_pos_ead.png", dpi=150, bbox_inches="tight")
plt.show()

pares = []
for i, c1 in enumerate(FEATURES):
    for c2 in FEATURES[i + 1:]:
        v = corr.loc[c1, c2]
        if pd.notna(v) and abs(v) >= 0.70:
            pares.append({"Feature 1": c1, "Feature 2": c2, "r": round(v, 4)})

print("=" * 66)
print("MULTICOLINEARIDADE ALTA NA POS EAD (|r| >= 0.70)")
print("=" * 66)
if pares:
    dfc = pd.DataFrame(pares)
    print(dfc.reindex(dfc.r.abs().sort_values(ascending=False).index).to_string(index=False))
    print("\nNao removidas: com regressao logistica L2 a multicolinearidade infla o erro")
    print("dos coeficientes, mas nao degrada o ranqueamento, que e o uso pretendido.")
else:
    print("Nenhum par com multicolinearidade severa.")

## 15. Auditoria de Viés por Sexo

`SEXO` está fora do `X`, mas remover a coluna **não garante** ausência de viés: no
híbrido o desvio sobreviveu à remoção, entrando por proxy via `CURSO`. Medimos sobre as
predições out-of-fold.

`UNIDADE_FISICA` não é auditável nesta base (concentração quase total num único valor),
mesma limitação já registrada no híbrido.

In [ ]:
aud = df_model.loc[OK, ["SEXO"]].copy()
aud["EVADIU"] = y_test
aud["score"] = y_score
corte_topo = np.sort(y_score)[::-1][k10 - 1]
aud["no_topo"] = (aud.score >= corte_topo).astype(int)

linhas = []
for sexo, g in aud.groupby("SEXO"):
    ev = int(g.EVADIU.sum())
    linhas.append({
        "SEXO": sexo, "Alunos": len(g), "Evasores": ev,
        "Evasão real": f"{g.EVADIU.mean():.2%}",
        "recall@10%": f"{g.loc[g.EVADIU == 1, 'no_topo'].mean():.1%}" if ev else "n/d",
    })
print(pd.DataFrame(linhas).to_string(index=False))

print("\n" + "=" * 66)
print("LEITURA: os denominadores sao pequenos demais para concluir.")
print("Um unico evasor mudando de lado move o recall do grupo em dezenas de")
print("pontos. A auditoria fica registrada para ser refeita quando a base crescer.")
print("=" * 66)

## Conclusão e próximos passos

**Não publicar modelo para a pós EAD nesta rodada** — e a razão é mais específica do
que "o desempenho é baixo".

Há sinal real: modelo e regra simples ficam ambos acima do acaso com p-valor de 0,017.
A atividade no AVA nos primeiros 30 dias **realmente** carrega informação sobre evasão.
O problema é outro: **o modelo acerta exatamente os mesmos 6 evasores que a regra de um
único critério**. Vinte variáveis, um pipeline e um artefato `.pkl` para entregar o que
`ORDER BY AVA_DIAS_ATIVOS` entrega.

Um modelo só se justifica quando supera a heurística que ele pretende substituir. Aqui
não supera — e a heurística ainda tem a vantagem de ser auditável pela coordenação e
explicável ao aluno. Publicar o `.pkl` como produto adicionaria opacidade sem adicionar
capacidade.

O gargalo é dado, não método:

1. **Poucos positivos.** 34 evasores no público acionável — 1,7 por feature, contra as
   ~10 que a prática recomenda.
2. **Pouco histórico.** O AVA só vira dado de produção em ago/2025, o que limita a oito
   meses de ingresso com alvo fechado.
3. **A pós é estruturalmente diferente.** Calendário anual, `ERP_HISTMATRICULA` cobrindo
   10–40% e módulos que fecham depois do corte. As features acadêmicas — as mais fortes
   na graduação — não existem aqui.

### Em ordem

- **Operar a regra simples agora.** "Sem acesso ao AVA nos primeiros 30 dias" separa
  8,6% de evasão contra 3,8%, cobrindo 23,5% dos evasores em 12% da base — e, para
  ordenar a fila inteira, `AVA_DIAS_ATIVOS` crescente entrega o mesmo `recall@10%` do
  modelo. Não é modelo, mas é auditável, explicável ao aluno e não finge precisão que
  não tem.
- **Critério objetivo para a próxima rodada:** só publicar modelo quando ele superar a
  regra de um critério por margem que sobreviva ao intervalo de confiança. Este é o
  teste que faltava no projeto e que esta rodada passa a exigir.
- **Registrar o desfecho das abordagens.** Sem esse retorno, a próxima rodada repete
  este impasse. É o investimento de maior retorno.
- **Remedir por volta de mar/2027**, quando a base deve dobrar.
- **Buscar sinal de engajamento mais fino** no AVA (entrega de atividade, acesso a
  material, fórum) — contagem de sessão é medida grosseira.